# T6: 2026 R02 中国GP — FP予選シミュレーション vs 実際の予選結果

## 概要
R02中国GPはスプリントウィークエンドのためFP1のみ。  
FP1の予選シミュレーションラップ（CLAUDE.md準拠の個別ラップベース識別）から  
各ドライバーのFP予測ラップタイムを算出し、実際の予選結果と比較する。

### 予選シミュレーション識別ロジック（CLAUDE.md準拠）
1. Softタイヤで記録されたラップ
2. アウトラップ（PitOutTime_secが存在）を除外
3. TyreLife <= 8（新品〜浅い使用状態）
4. セッション最速の103%以内（本気アタックラップ）

## セットアップ & データ読み込み

In [ ]:
import matplotlib
matplotlib.use('Agg')  # GUIなし環境対応

import csv
import os
import math
import statistics
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from scipy import stats

# ===== パス設定 =====
BASE_DIR = '/Volumes/lyssr_workspace/2026_1_4/Motorsports-Visualised'
FP_CSV = os.path.join(BASE_DIR, 'data/2026_R02_China/export/fp_laps.csv')
QUALI_CSV = os.path.join(BASE_DIR, 'data/2026_R02_China/export/quali_laps.csv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'notebooks/output')
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ===== グラフスタイル =====
STYLE = {
    'bg_color': '#1a1a2e',
    'text_color': '#ffffff',
    'grid_color': '#333355',
    'figsize': (12, 6.75),
    'title_size': 18,
    'label_size': 12,
}

# ===== CSV読み込みヘルパー =====
def load_csv(path):
    """CSVをdict listとして読み込む"""
    with open(path, encoding='utf-8') as f:
        return list(csv.DictReader(f))

def to_float(val, default=None):
    """文字列→float変換（空文字・NaN対応）"""
    try:
        v = float(val)
        return v if not math.isnan(v) else default
    except (TypeError, ValueError):
        return default

# FPデータ読み込み
fp_rows = load_csv(FP_CSV)
quali_rows = load_csv(QUALI_CSV)

# セッション種別を動的に確認
sessions_found = sorted(set(r['Session'] for r in fp_rows))
print(f"総FPラップ数: {len(fp_rows)}")
print(f"検出されたセッション: {sessions_found}")
print(f"予選総ラップ数: {len(quali_rows)}")